In [ ]:
#update method in BallTracker takes detection results, converts bounding boxes to center points, and stores them in a buffer. It then calculates the average ball position (centroid) and selects the ball closest to it, assuming there's only one ball on the field and its movement is physically realistic.

import supervision as sv
import numpy as np

from collections import deque

class BallTracker:
    def __init__(self, buffer_size: int = 10): # 
        self.buffer = deque(maxlen=buffer_size)

    def update(self, detections: sv.Detections) -> sv.Detections:
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        self.buffer.append(xy)

        if len(detections) == 0:
            return detections

        centroid = np.mean(np.concatenate(self.buffer), axis=0)
        distances = np.linalg.norm(xy - centroid, axis=1)
        index = np.argmin(distances)
        return detections[[index]]


In [ ]:
# Weighted averaging giving prevence to recent ones.
import supervision as sv
import numpy as np
from collections import deque

"""Exponential Decay Weights: Recent frames get exponentially higher weights using a decay_factor parameter (default 0.9)
Normalized Weights: Weights sum to 1.0, ensuring the weighted average stays within reasonable bounds
Weighted Centroid Calculation: Instead of simple mean, uses np.sum(weights * positions)
Configurable Decay: You can adjust how much recent vs. historical data matters via decay_factor"""
class BallTracker:
    def __init__(self, buffer_size: int = 10, decay_factor: float = 0.9):
        """
        Initialize the ball tracker with weighted averaging.
        
        Args:
            buffer_size: Maximum number of frames to keep in buffer
            decay_factor: Weight decay factor (0 < decay_factor < 1). 
                         Higher values give more weight to recent frames.
        """
        self.buffer = deque(maxlen=buffer_size)
        self.decay_factor = decay_factor
        self.buffer_size = buffer_size

    def _calculate_weights(self, buffer_length: int) -> np.ndarray:
        """
        Calculate exponential decay weights for the buffer.
        Most recent frame gets weight 1.0, earlier frames get progressively smaller weights.
        """
        if buffer_length == 0:
            return np.array([])
        
        # Create weights: [decay^(n-1), decay^(n-2), ..., decay^1, decay^0]
        # where n is buffer_length and decay^0 = 1.0 for the most recent frame
        weights = np.array([self.decay_factor ** (buffer_length - 1 - i) 
                           for i in range(buffer_length)])
        
        # Normalize weights to sum to 1
        return weights / np.sum(weights)

    def _calculate_weighted_centroid(self) -> np.ndarray:
        """
        Calculate weighted centroid from buffer using exponential decay weights.
        """
        if len(self.buffer) == 0:
            return np.array([0, 0])
        
        # Get weights for current buffer
        weights = self._calculate_weights(len(self.buffer))
        
        # Calculate weighted average for each frame's detections
        weighted_positions = []
        
        for i, positions in enumerate(self.buffer):
            if len(positions) > 0:
                # If multiple detections in a frame, take their mean
                frame_centroid = np.mean(positions, axis=0)
                weighted_positions.append(weights[i] * frame_centroid)
        
        if len(weighted_positions) == 0:
            return np.array([0, 0])
        
        # Sum all weighted positions to get final weighted centroid
        return np.sum(weighted_positions, axis=0)

    def update(self, detections: sv.Detections) -> sv.Detections:
        """
        Update tracker with new detections and return the best detection.
        """
        # Get center coordinates of current detections
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
        # Add current detections to buffer
        self.buffer.append(xy)
        
        # If no detections, return empty
        if len(detections) == 0:
            return detections
        
        # Calculate weighted centroid from historical data
        weighted_centroid = self._calculate_weighted_centroid()
        
        # Find detection closest to weighted centroid
        distances = np.linalg.norm(xy - weighted_centroid, axis=1)
        index = np.argmin(distances)
        
        return detections[[index]]

    def get_predicted_position(self) -> np.ndarray:
        """
        Get the current predicted position (weighted centroid) without new detections.
        Useful for visualization or when no detections are found.
        """
        return self._calculate_weighted_centroid()
    
    def reset(self):
        """Reset the tracker buffer."""
        self.buffer.clear()

In [ ]:

class PhysicsBasedFilter:
    def __init__(self, max_velocity_change=50, max_acceleration=20):
        self.position_history = deque(maxlen=3)
        self.max_velocity_change = max_velocity_change
        self.max_acceleration = max_acceleration
    
    def is_motion_consistent(self, new_position):
        if len(self.position_history) < 2:
            return True
        
        # Calculate current and previous velocities
        prev_pos = self.position_history[-1]
        prev_prev_pos = self.position_history[-2]
        
        prev_velocity = prev_pos - prev_prev_pos
        current_velocity = new_position - prev_pos
        
        # Check velocity change (acceleration)
        velocity_change = np.linalg.norm(current_velocity - prev_velocity)
        
        return velocity_change < self.max_velocity_change



In [ ]:
class SizeConsistentTracker:
    def __init__(self, size_tolerance=0.3):
        self.size_history = deque(maxlen=10)
        self.size_tolerance = size_tolerance
    
    def filter_by_size(self, detections):
        if len(self.size_history) == 0:
            return detections
        
        expected_area = np.mean(self.size_history)
        areas = (detections.xyxy[:, 2] - detections.xyxy[:, 0]) * \
                (detections.xyxy[:, 3] - detections.xyxy[:, 1])
        
        size_ratios = areas / expected_area
        valid_mask = (size_ratios > (1 - self.size_tolerance)) & \
                    (size_ratios < (1 + self.size_tolerance))
        
        return detections[valid_mask]

In [ ]:
# Dynamic thresholding based on detection quality
def filter_by_confidence(detections, min_confidence=0.5, adaptive=True):
    if adaptive and len(detections) > 1:
        # Use mean + std to set adaptive threshold
        scores = detections.confidence
        threshold = np.mean(scores) - 0.5 * np.std(scores)
        threshold = max(threshold, min_confidence)
    else:
        threshold = min_confidence
    
    return detections[detections.confidence > threshold]

In [ ]:
class TemporalConsistencyFilter:
    def __init__(self, min_track_length=3, max_gap=2):
        self.track_states = {}  # track_id -> consecutive_frames
        self.gap_counts = {}    # track_id -> gap_count
        self.min_track_length = min_track_length
        self.max_gap = max_gap
    
    def filter_tracks(self, detections, track_ids):
        valid_indices = []
        
        for i, track_id in enumerate(track_ids):
            # Update track state
            if track_id in self.track_states:
                self.track_states[track_id] += 1
                self.gap_counts[track_id] = 0
            else:
                self.track_states[track_id] = 1
                self.gap_counts[track_id] = 0
            
            # Keep track if it's been consistent long enough
            if self.track_states[track_id] >= self.min_track_length:
                valid_indices.append(i)
        
        return detections[valid_indices]

In [ ]:
class ContextualFilter:
    def __init__(self, field_boundaries=None, player_positions=None):
        self.field_boundaries = field_boundaries
        self.player_positions = player_positions
    
    def filter_by_context(self, detections):
        positions = detections.get_anchors_coordinates(sv.Position.CENTER)
        valid_mask = np.ones(len(detections), dtype=bool)
        
        # Remove detections outside field boundaries
        if self.field_boundaries:
            x_min, y_min, x_max, y_max = self.field_boundaries
            valid_mask &= (positions[:, 0] >= x_min) & (positions[:, 0] <= x_max)
            valid_mask &= (positions[:, 1] >= y_min) & (positions[:, 1] <= y_max)
        
        # Remove detections too close to players (likely false positives)
        if self.player_positions is not None:
            for player_pos in self.player_positions:
                distances = np.linalg.norm(positions - player_pos, axis=1)
                valid_mask &= distances > 30  # Minimum distance from players
        
        return detections[valid_mask]

In [ ]:
#Train a model maybe LSTM to identify real balls from false positives: based on features such as area, confidence, and making other features. 

In [ ]:
#smoothning based on tracking Ids, built in supervision.
import supervision as sv

from ultralytics import YOLO

video_info = sv.VideoInfo.from_video_path(video_path=<SOURCE_FILE_PATH>)
frame_generator = sv.get_video_frames_generator(source_path=<SOURCE_FILE_PATH>)

model = YOLO(<MODEL_PATH>)
tracker = sv.ByteTrack(frame_rate=video_info.fps)
smoother = sv.DetectionsSmoother()

box_annotator = sv.BoxAnnotator()

with sv.VideoSink(<TARGET_FILE_PATH>, video_info=video_info) as sink:
    for frame in frame_generator:
        result = model(frame)[0]
        detections = sv.Detections.from_ultralytics(result)
        detections = tracker.update_with_detections(detections)
        detections = smoother.update_with_detections(detections)

        annotated_frame = box_annotator.annotate(frame.copy(), detections)
        sink.write_frame(annotated_frame)